# MNIST 3D Manifold 시각화

병목 레이어(3노드)를 통해 10개 클래스가 3차원 공간에 어떻게 매핑되는지 시각화합니다.

**모델 구조:** `784 → 512 → 256 → 128 → [3] → 10`

> **권장 런타임:** 상단 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 실행하세요.

## 0. 패키지 설치

In [ ]:
!pip install -q plotly

## 1. 임포트 및 하이퍼파라미터

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import plotly.graph_objects as go
import plotly.io as pio

# Colab 인라인 렌더링 설정
pio.renderers.default = "colab"

print(f"TensorFlow 버전: {tf.__version__}")
print(f"GPU 사용 가능: {len(tf.config.list_physical_devices('GPU')) > 0}")

# ── 하이퍼파라미터 ──
BATCH_SIZE  = 256
EPOCHS      = 15
LR          = 1e-3
BOTTLENECK  = 3       # 3D manifold 병목 노드 수
VIZ_SAMPLES = 5000   # 시각화에 사용할 테스트 샘플 수
MODEL_PATH  = "best_model.keras"

## 2. 데이터 로드 및 전처리

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 정규화 (MNIST 평균/표준편차) 및 평탄화: (N, 28, 28) → (N, 784)
mean, std = 0.1307, 0.3081
x_train = (x_train.reshape(-1, 784).astype("float32") / 255.0 - mean) / std
x_test  = (x_test.reshape(-1, 784).astype("float32") / 255.0 - mean) / std

print(f"학습 샘플: {len(x_train):,}  /  테스트 샘플: {len(x_test):,}")
print(f"x_train shape: {x_train.shape}, dtype: {x_train.dtype}")

## 3. 모델 정의 (Keras Functional API)

```
784 → Dense(512)+BN+ReLU → Dense(256)+BN+ReLU → Dense(128)+BN+ReLU
    → Dense(3) [병목]  → Dense(10, softmax)
```

In [ ]:
def build_model(bottleneck_dim=3):
    inputs = keras.Input(shape=(784,), name="input")

    x = layers.Dense(512)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Dense(128)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    bottleneck = layers.Dense(bottleneck_dim, name="bottleneck")(x)  # ← 3D 병목
    outputs    = layers.Dense(10, activation="softmax", name="classifier")(bottleneck)

    model         = keras.Model(inputs, outputs,    name="MNISTManifoldNet")
    feature_model = keras.Model(inputs, bottleneck, name="FeatureExtractor")
    return model, feature_model


model, feature_model = build_model(bottleneck_dim=BOTTLENECK)
print(f"파라미터 수: {model.count_params():,}  /  병목 차원: {BOTTLENECK}D")
model.summary()

## 4. 학습

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, verbose=1,
    ),
]

history = model.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
    verbose=1,
)

# 최적 가중치 로드 후 최종 평가
model.load_weights(MODEL_PATH)
_, best_acc = model.evaluate(x_test, y_test, batch_size=BATCH_SIZE, verbose=0)
print(f"\n✅ 최고 테스트 정확도: {best_acc*100:.2f}%  (3노드 병목 제약)")

## 5. 학습 곡선 확인

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

epochs_range = list(range(1, len(history.history["loss"]) + 1))

fig_hist = make_subplots(rows=1, cols=2, subplot_titles=("손실 (Loss)", "정확도 (Accuracy)"))

for col, (train_key, val_key, yaxis_title) in enumerate(
    [("loss", "val_loss", "Loss"), ("accuracy", "val_accuracy", "Accuracy")], start=1
):
    fig_hist.add_trace(go.Scatter(x=epochs_range, y=history.history[train_key],
                                  name="Train", line=dict(color="#3498DB")), row=1, col=col)
    fig_hist.add_trace(go.Scatter(x=epochs_range, y=history.history[val_key],
                                  name="Validation", line=dict(color="#E74C3C", dash="dash")), row=1, col=col)

fig_hist.update_layout(title="학습 곡선", showlegend=True, height=400)
fig_hist.show()

## 6. 병목 레이어 3D 특징 추출

In [ ]:
features = feature_model.predict(x_test[:VIZ_SAMPLES], batch_size=BATCH_SIZE, verbose=0)
labels   = y_test[:VIZ_SAMPLES]

assert features.shape[1] == 3, f"병목 출력 shape 오류: {features.shape}"
print(f"추출된 특징 shape: {features.shape}")

print("\n클래스별 샘플 수:")
for d in range(10):
    print(f"  숫자 {d}: {(labels == d).sum():>5}개")

## 7. 인터랙티브 3D 시각화 (Plotly)

마우스로 **회전 / 줌 / 호버**가 가능합니다.

In [ ]:
COLORS = [
    "#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6",
    "#1ABC9C", "#E67E22", "#34495E", "#E91E63", "#00BCD4",
]

fig = go.Figure()

for digit in range(10):
    mask = labels == digit
    fig.add_trace(go.Scatter3d(
        x=features[mask, 0],
        y=features[mask, 1],
        z=features[mask, 2],
        mode="markers",
        name=f"숫자 {digit}",
        marker=dict(size=2.5, color=COLORS[digit], opacity=0.7, line=dict(width=0)),
        hovertemplate=(
            f"<b>클래스: {digit}</b><br>"
            "x: %{x:.3f}<br>y: %{y:.3f}<br>z: %{z:.3f}<extra></extra>"
        ),
    ))

fig.update_layout(
    title=dict(
        text=(
            f"MNIST 3D Manifold 시각화<br>"
            f"<sup>병목 레이어 3노드 → 3차원 공간 매핑 "
            f"(테스트 정확도: {best_acc*100:.2f}%)</sup>"
        ),
        x=0.5, font=dict(size=18),
    ),
    scene=dict(
        xaxis=dict(title="차원 1", backgroundcolor="#F8F9FA", gridcolor="white"),
        yaxis=dict(title="차원 2", backgroundcolor="#F8F9FA", gridcolor="white"),
        zaxis=dict(title="차원 3", backgroundcolor="#F8F9FA", gridcolor="white"),
        bgcolor="#F0F2F5",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
    ),
    legend=dict(title="숫자 클래스", itemsizing="constant", font=dict(size=13)),
    width=900, height=700,
    margin=dict(l=0, r=0, t=80, b=0),
    paper_bgcolor="white",
)

fig.show()

# HTML 파일로도 저장 (Google Drive에 저장하려면 경로를 변경하세요)
fig.write_html("mnist_manifold_3d.html", include_plotlyjs="cdn")
print("✅ mnist_manifold_3d.html 저장 완료")